In [1]:
import dlt
from itertools import islice
from dlt.sources.rest_api import rest_api_source
import duckdb

## 🔗 Step 2: Define the API Source (Open Library)

In [2]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })

## 🔧 Step 3: Create the dlt Pipeline

In [3]:
pipeline = dlt.pipeline(
    pipeline_name="open_library",
    destination="duckdb",       # le type de destination
    dataset_name="library",
    progress="log",
    dev_mode=False           # si tu veux réécrire tout
)

In [4]:
extract_info = pipeline.extract(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 171.55 MB (38.10%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.95s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 8065969.23/s
Memory usage: 173.67 MB (38.10%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.27s | Rate: 0.00/s
books: 500  | Time: 1.32s | Rate: 378.00/s
Memory usage: 174.79 MB (38.00%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.51s | Rate: 0.00/s
books: 800  | Time: 2.57s | Rate: 311.66/s
Memory usage: 176.42 MB (38.10%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.67s | Rate: 0.

## ⬇️ Step 4: Extract

Extract means:
- dlt sends requests to the Open Library API
- the raw JSON responses are downloaded
- the results are stored in dlt’s local working folder

At this stage, the data is **not** in DuckDB yet. We are just confirming that we successfully pulled data from the API.

In [5]:
extract_info = pipeline.extract(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 184.14 MB (41.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.65s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 9986438.10/s
Memory usage: 184.18 MB (41.90%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.70s | Rate: 0.00/s
books: 400  | Time: 1.06s | Rate: 378.65/s
Memory usage: 184.25 MB (41.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.22s | Rate: 0.00/s
books: 600  | Time: 2.58s | Rate: 232.89/s
Memory usage: 185.05 MB (41.90%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.36s | Rate: 0.

In [6]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['books']
Tables: ['books']
Load ID: 1774132562.772938

Resource: books
rows extracted: 3768



## 🔄 Step 5: Normalize

During normalization, dlt does three key things:

### 1. Adds Tracking Columns to the Main Table
dlt adds special columns to every table:
- `_dlt_id`: A unique identifier for each row
- `_dlt_load_id`: Links each row to the load job that created it

### 2. Flattens Nested Data into Child Tables
dlt flattens these nested structures into separate **child tables** with names like:
- `books__author_name`
- `books__author_key`
- `books__language`

Each child table has a `_dlt_parent_id` column that references `_dlt_id` in the parent table. This is how dlt maintains relationships.

### 3. Creates Metadata Tables
dlt also creates internal tables to track pipeline state:
- `_dlt_loads`: Tracks load history (when data was loaded, status)
- `_dlt_pipeline_state`: Stores pipeline state for incremental loading
- `_dlt_version`: Tracks schema versions

In [7]:
normalize_info = pipeline.normalize()

------------------- Normalize rest_api in 1774132542.2897227 -------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 187.42 MB (40.10%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1774132542.2897227 -------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 187.42 MB (40.10%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1774132542.2897227 -------------------
Files: 12/1 (1200.0%) | Time: 0.83s | Rate: 14.54/s
Items: 23366  | Time: 0.82s | Rate: 28324.84/s
Memory usage: 190.32 MB (40.10%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1774132562.772938 --------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 190.32 MB (40.10%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1774132562.772938 --------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 190.

In [8]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")

Load ID: 1774132562.772938

Tables created/updated:
  - books: 3768 rows
  - books__author_key: 4633 rows
  - books__author_name: 4633 rows
  - books__ia: 6030 rows
  - books__ia_collection: 3670 rows
  - books__language: 4097 rows
  - books__series_key: 9 rows
  - books__series_name: 9 rows
  - books__series_position: 9 rows
  - books__id_standard_ebooks: 23 rows
  - books__id_librivox: 124 rows
  - books__id_project_gutenberg: 110 rows


In [9]:
# Display schema 
pipeline.default_schema

<dlt.Schema(name='rest_api', version=5, tables=['_dlt_version', '_dlt_loads', 'books', '_dlt_pipeline_state', 'books__author_key', 'books__author_name', 'books__language', 'books__ia', 'books__ia_collection', 'books__id_project_gutenberg', 'books__id_librivox', 'books__id_standard_ebooks', 'books__series_key', 'books__series_name', 'books__series_position'], version_hash='OKGqzSJEa+e9I9QaeeI/h/v1JAAzvB/d2WFklS9XNbU=')>

## 📤 Step 6: 

Load means:

    dlt creates tables in DuckDB (if they do not already exist)
    the normalized rows are inserted into those tables
    the pipeline records the load in its internal tracking tables


In [10]:
load_info = pipeline.load()

--------------------- Load rest_api in 1774132542.2897227 ----------------------
Jobs: 0/12 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 190.75 MB (40.00%) | CPU usage: 0.00%

--------------------- Load rest_api in 1774132542.2897227 ----------------------
Jobs: 9/12 (75.0%) | Time: 1.02s | Rate: 8.85/s
Memory usage: 287.89 MB (40.40%) | CPU usage: 0.00%

--------------------- Load rest_api in 1774132542.2897227 ----------------------
Jobs: 12/12 (100.0%) | Time: 1.77s | Rate: 6.78/s
Memory usage: 207.20 MB (40.20%) | CPU usage: 0.00%

---------------------- Load rest_api in 1774132562.772938 ----------------------
Jobs: 0/12 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 207.20 MB (40.20%) | CPU usage: 0.00%

---------------------- Load rest_api in 1774132562.772938 ----------------------
Jobs: 9/12 (75.0%) | Time: 1.02s | Rate: 8.80/s
Memory usage: 310.06 MB (40.70%) | CPU usage: 0.00%

---------------------- Load rest_api in 1774132562.772938 ----------------------
Jobs: 12/

## 🚀 Step 7: Run the Full Pipeline

<p>
  <code>pipeline.run()</code> simply combines the three steps we already executed manually:
</p>
<ol>
  <li><strong>Extract</strong> – fetch data from the Open Library API</li>
  <li><strong>Normalize</strong> – convert nested JSON into relational tables</li>
  <li><strong>Load</strong> – write those tables into DuckDB</li>
</ol>
<p>In other words, this:</p>
<pre><code>pipeline.run(source)</code></pre>
<p>is equivalent to:</p>
<pre><code>pipeline.extract(source)
pipeline.normalize()
pipeline.load()</code></pre>

In [11]:
# Exécuter le pipeline
print("🚀 Chargement des données...")

load_info = pipeline.run(openlibrary_source(query="data ingestion dlt"))

print("✅ Chargement terminé!")
print(f"📊 Dataset: {load_info.dataset_name}")
print(f"🗄️  Base: open_library.duckdb")

🚀 Chargement des données...
------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 199.95 MB (40.30%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.87s | Rate: 0.00/s
books: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 199.96 MB (40.20%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 1/1 (100.0%) | Time: 0.88s | Rate: 1.14/s
books: 0  | Time: 0.01s | Rate: 0.00/s
Memory usage: 199.96 MB (40.20%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1774132587.2690804 -------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 200.00 MB (40.20%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1774132587.2690804 -------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Ra

## 🔎 Step 8: Inspect the Loaded Data


In [12]:
# List available tables
ds = pipeline.dataset()
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__language',
 'books__ia',
 'books__ia_collection',
 'books__id_project_gutenberg',
 'books__id_librivox',
 'books__id_standard_ebooks',
 'books__series_key',
 'books__series_name',
 'books__series_position',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [13]:
# main table
df = ds.books.df()      
df.head(3)

,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,public_scan_b,title,_dlt_load_id,_dlt_id,lending_edition_s,lending_identifier_s,subtitle


In [14]:
import duckdb

# Se connecter
con = duckdb.connect("open_library.duckdb")

# 1. Voir les schémas
print("📁 SCHÉMAS:")
schemas = con.execute("SHOW SCHEMAS").fetchdf()
print(schemas)


📁 SCHÉMAS:
  database_name schema_name  current
0  open_library     library    False
1  open_library        main     True


In [15]:

# 2. Voir les tables
print("\n📊 TABLES:")
tables = con.execute("SHOW TABLES FROM library").fetchdf()
print(tables)



📊 TABLES:
                           name
0                    _dlt_loads
1           _dlt_pipeline_state
2                  _dlt_version
3                         books
4             books__author_key
5            books__author_name
6                     books__ia
7          books__ia_collection
8            books__id_librivox
9   books__id_project_gutenberg
10    books__id_standard_ebooks
11              books__language
12            books__series_key
13           books__series_name
14       books__series_position


In [16]:
# 3. Compter les livres
count = con.execute("SELECT COUNT(*) as total FROM library.books").fetchdf()
print(f"\n📚 Total de livres: {count['total'][0]}")


📚 Total de livres: 0


In [17]:
# 4. Aperçu des données
print("\n📖 APERÇU DES LIVRES:")
books_df = con.execute("""
    SELECT 
        key,
        title,
        first_publish_year
    FROM library.books 
    LIMIT 10
""").fetchdf()
print(books_df)


📖 APERÇU DES LIVRES:
Empty DataFrame
Columns: [key, title, first_publish_year]
Index: []


In [18]:
# 5. Structure de la table
print("\n🏗️  STRUCTURE:")
structure = con.execute("DESCRIBE library.books").fetchdf()
print(structure)

con.close()


🏗️  STRUCTURE:
             column_name column_type null   key default extra
0      cover_edition_key     VARCHAR  YES  None    None  None
1                cover_i      BIGINT  YES  None    None  None
2           ebook_access     VARCHAR  YES  None    None  None
3          edition_count      BIGINT  YES  None    None  None
4     first_publish_year      BIGINT  YES  None    None  None
5           has_fulltext     BOOLEAN  YES  None    None  None
6                    key     VARCHAR   NO  None    None  None
7          public_scan_b     BOOLEAN  YES  None    None  None
8                  title     VARCHAR  YES  None    None  None
9           _dlt_load_id     VARCHAR   NO  None    None  None
10               _dlt_id     VARCHAR   NO  None    None  None
11     lending_edition_s     VARCHAR  YES  None    None  None
12  lending_identifier_s     VARCHAR  YES  None    None  None
13              subtitle     VARCHAR  YES  None    None  None


In [19]:
con = duckdb.connect("open_library.duckdb")

# Statistiques
print("📈 STATISTIQUES:")
stats = con.execute("""
    SELECT 
        COUNT(*) as total_books,
        COUNT(DISTINCT title) as unique_titles,
        MIN(first_publish_year) as oldest_year,
        MAX(first_publish_year) as newest_year,
    FROM library.books
    WHERE first_publish_year IS NOT NULL
""").fetchdf()
print(stats)

# Top auteurs (si dlt a créé la table liée)
print("\n👤 TOP AUTEURS:")
try:
    authors = con.execute("""
        SELECT 
            value as author,
            COUNT(*) as book_count
        FROM library.books__author_name
        GROUP BY author
        ORDER BY book_count DESC
        LIMIT 10
    """).fetchdf()
    print(authors)
except:
    print("Table author_name pas disponible")

con.close()

📈 STATISTIQUES:
   total_books  unique_titles  oldest_year  newest_year
0            0              0         <NA>         <NA>

👤 TOP AUTEURS:
Empty DataFrame
Columns: [author, book_count]
Index: []
